# Aerial Crime (Violence) Detection — Training & Evaluation

This notebook trains a video classifier (MobileNetV2 + Bidirectional LSTM, a.k.a. "MoBiLSTM") on your `Violence_Dataset` and reports **Accuracy, Precision, Recall, and F1-score** on the test split.

Expected dataset structure (upload/zip it to your Drive):
```
Violence_Dataset/
  train/
    Violence/
    NonViolence/
  val/
    Violence/
    NonViolence/
  test/
    Violence/
    NonViolence/
```

**Runtime:** Runtime > Change runtime type > GPU (T4).

## 1. Setup

In [ ]:
!pip install -q opencv-python-headless scikit-learn seaborn

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, TimeDistributed, GlobalAveragePooling2D,
                                      Bidirectional, LSTM, Dense, Dropout)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS to point to where you uploaded/unzipped Violence_Dataset in your Drive
DATASET_DIR = '/content/drive/MyDrive/Violence_Dataset'

assert os.path.exists(DATASET_DIR), f'Dataset not found at {DATASET_DIR} — update DATASET_DIR above.'
print(os.listdir(DATASET_DIR))

## 2. Configuration

In [ ]:
IMAGE_HEIGHT, IMAGE_WIDTH = 64, 64   # frame size fed to the CNN (kept small for speed)
SEQUENCE_LENGTH = 16                 # number of frames sampled per video clip
CLASSES_LIST = ['NonViolence', 'Violence']   # index 0 / 1 -- must match folder names
BATCH_SIZE = 8
EPOCHS = 30

## 3. Frame extraction & dataset loading

In [ ]:
def frames_extraction(video_path):
    """Uniformly samples SEQUENCE_LENGTH frames from a video, resizes and normalizes them."""
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    if video_frames_count <= 0:
        video_reader.release()
        return frames_list

    skip_frames_window = max(int(video_frames_count / SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break
        resized_frame = cv2.resize(frame, (IMAGE_WIDTH, IMAGE_HEIGHT))
        normalized_frame = resized_frame / 255.0
        frames_list.append(normalized_frame)

    video_reader.release()
    return frames_list

In [ ]:
def create_dataset(split):
    """split: 'train' | 'val' | 'test'. Returns (features, labels, paths)."""
    features, labels, video_paths = [], [], []
    split_dir = os.path.join(DATASET_DIR, split)

    for class_index, class_name in enumerate(CLASSES_LIST):
        class_dir = os.path.join(split_dir, class_name)
        files_list = [f for f in os.listdir(class_dir) if f.lower().endswith(('.mp4', '.avi', '.mov'))]
        print(f'[{split}] Extracting {len(files_list)} videos from class: {class_name}')

        for file_name in files_list:
            video_file_path = os.path.join(class_dir, file_name)
            frames = frames_extraction(video_file_path)
            if len(frames) == SEQUENCE_LENGTH:
                features.append(frames)
                labels.append(class_index)
                video_paths.append(video_file_path)

    features = np.asarray(features, dtype=np.float32)
    labels = np.array(labels)
    return features, labels, video_paths

In [ ]:
features_train, labels_train, _ = create_dataset('train')
features_val, labels_val, _ = create_dataset('val')
features_test, labels_test, test_paths = create_dataset('test')

print('Train:', features_train.shape, labels_train.shape)
print('Val:  ', features_val.shape, labels_val.shape)
print('Test: ', features_test.shape, labels_test.shape)

In [ ]:
labels_train_ohe = to_categorical(labels_train, num_classes=len(CLASSES_LIST))
labels_val_ohe = to_categorical(labels_val, num_classes=len(CLASSES_LIST))
labels_test_ohe = to_categorical(labels_test, num_classes=len(CLASSES_LIST))

## 4. Model — MobileNetV2 (per-frame features) + Bidirectional LSTM (temporal)

In [ ]:
def create_model():
    mobilenet = MobileNetV2(include_top=False, weights='imagenet',
                             input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3))
    mobilenet.trainable = True
    # Freeze all but the last ~40 layers to fine-tune high-level features only
    for layer in mobilenet.layers[:-40]:
        layer.trainable = False

    model = Sequential([
        Input(shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3)),
        TimeDistributed(mobilenet),
        TimeDistributed(GlobalAveragePooling2D()),
        Dropout(0.25),
        Bidirectional(LSTM(32, return_sequences=False)),
        Dropout(0.25),
        Dense(256, activation='relu'),
        Dropout(0.25),
        Dense(len(CLASSES_LIST), activation='softmax')
    ])
    return model

model = create_model()
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=8, mode='min',
                                restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)

## 5. Train

In [ ]:
history = model.fit(
    x=features_train, y=labels_train_ohe,
    validation_data=(features_val, labels_val_ohe),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='train_loss')
axes[0].plot(history.history['val_loss'], label='val_loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train_acc')
axes[1].plot(history.history['val_accuracy'], label='val_acc')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.show()

## 6. Evaluate on test set — Accuracy, Precision, Recall, F1

In [ ]:
pred_probs = model.predict(features_test)
pred_labels = np.argmax(pred_probs, axis=1)

accuracy = accuracy_score(labels_test, pred_labels)
precision = precision_score(labels_test, pred_labels)
recall = recall_score(labels_test, pred_labels)
f1 = f1_score(labels_test, pred_labels)

print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print()
print(classification_report(labels_test, pred_labels, target_names=CLASSES_LIST))

In [ ]:
cm = confusion_matrix(labels_test, pred_labels)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES_LIST, yticklabels=CLASSES_LIST)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()

## 7. Save the trained model (for use in the Flask app)

In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/violence_detection_model.h5'
model.save(MODEL_SAVE_PATH)
print('Saved to', MODEL_SAVE_PATH)
print('Download this file and place it at flask_app/model/violence_detection_model.h5')